# 04 — Synthetic ground-truth Yamada recovery

Controlled validation at a **single fixed voxel resolution** $N=200$:

$$
G_{\rm true}\rightarrow F(G_{\rm true})\rightarrow V_{200}
\rightarrow \widehat G\rightarrow \Upsilon(\widehat G;A).
$$

Only connected, bridgeless, **exactly trivalent** ground-truth graphs are used.

The benchmark separates failure mechanisms rather than calling every mismatch a Yamada failure:

- `SKIP-GEOM`: tube thickness is too large relative to geometric clearance.
- `FAIL-DEG`: the recovered graph contains a vertex of degree $>3$; Yamada is not called.
- `FAIL-GRAPH`: the recovered abstract multigraph is not isomorphic to the ground truth; Yamada is not called.
- `FAIL-EMBED`: the abstract graph is correct but the recovered Yamada polynomial differs.
- `ERROR-RECOVER` / `ERROR-YAMADA`: execution failures in the indicated stage.
- `PASS`: abstract graph and Yamada polynomial are both recovered.

No plots are produced. All diagnostics are printed.


In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
import hashlib, json, sys
import networkx as nx
import numpy as np
import sympy as sp
from skimage.morphology import ball, dilation, skeletonize

ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'pyproject.toml').exists(): ROOT=ROOT.parent
SRC=ROOT/'src'
if not (SRC/'knotted_graph').exists(): raise RuntimeError('Run inside the KnottedGraph checkout.')
if str(SRC) not in sys.path: sys.path.insert(0,str(SRC))

import knotted_graph
from knotted_graph.core import contract_short_edges, remove_leaf_nodes, simplify_edges, smooth_edges
from knotted_graph.extraction import skeleton_image_to_graph
from knotted_graph.projection import compute_yamada_polynomial

kg_path=Path(knotted_graph.__file__).resolve()
if SRC not in kg_path.parents: raise RuntimeError(f'Stale knotted_graph import: {kg_path}')

A=sp.Symbol('A')
BOUND=1.35
N=200
RADII=[1,2,3]
TRANSFORMS=['identity','rotate','affine']
CLEARANCE_FRACTION=0.40
JUNCTION_CONTRACT_VOXELS=2.5
SMOOTH_VOXELS=2.0
PROJECTION_SAMPLES=16
CACHE_SCHEMA='synthetic-ground-truth-n200-v6'
CHECKPOINT=ROOT/'User_guide'/'benchmarks'/'synthetic_ground_truth_results_n200_v6.jsonl'
RESUME=True
DX=2*BOUND/(N-1)
print('KnottedGraph:',kg_path)
print(f'N={N}, dx={DX:.6f}, radii={RADII}, transforms={TRANSFORMS}')
print(f'cleanup: contract < {JUNCTION_CONTRACT_VOXELS:.1f} voxels ({JUNCTION_CONTRACT_VOXELS*DX:.6f} world), smooth epsilon={SMOOTH_VOXELS:.1f} voxels')


In [ ]:
@dataclass
class Case:
    name:str
    graph:nx.MultiGraph
    radius_cap:float

def normalize(X,scale=.72):
    X=np.asarray(X,float); X-=X.mean(0)
    return X*(scale/np.max(np.linalg.norm(X,axis=1)))

def embedded_graph(pos,edges):
    H=nx.MultiGraph()
    for n,p in pos.items(): H.add_node(n,pos=np.asarray(p,float))
    for u,v,P in edges: H.add_edge(u,v,pts=np.asarray(P,float))
    return H

def theta_case(name,bowed=False,n=500):
    t=np.linspace(0,1,n); x=-.72+1.44*t
    if bowed:
        C=[np.c_[x,-.58*np.sin(np.pi*t), .16*np.sin(2*np.pi*t)],
           np.c_[x, .10*np.sin(2*np.pi*t),-.10*np.sin(np.pi*t)],
           np.c_[x, .58*np.sin(np.pi*t),-.16*np.sin(2*np.pi*t)]]
    else:
        C=[np.c_[x,-.58*np.sin(np.pi*t),0*t],np.c_[x,0*t,0*t],np.c_[x,.58*np.sin(np.pi*t),0*t]]
    for P in C: P[0]=[-.72,0,0]; P[-1]=[.72,0,0]
    return Case(name,embedded_graph({'u':C[0][0],'v':C[0][-1]},[('u','v',P) for P in C]),.060)

def segdist(p1,q1,p2,q2):
    u=q1-p1; v=q2-p2; w=p1-p2
    a=u@u; b=u@v; c=v@v; d=u@w; e=v@w; D=a*c-b*b
    if D<1e-14: s=0.; t=np.clip(e/c if c>1e-14 else 0.,0,1)
    else: s=np.clip((b*e-c*d)/D,0,1); t=np.clip((a*e-b*d)/D,0,1)
    if a>1e-14: s=np.clip((b*t-d)/a,0,1)
    if c>1e-14: t=np.clip((b*s+e)/c,0,1)
    return float(np.linalg.norm(w+s*u-t*v))

def straight_clearance(G,P):
    E=list(G.edges()); best=np.inf
    for i,(u,v) in enumerate(E):
        for a,b in E[i+1:]:
            if {u,v}&{a,b}: continue
            best=min(best,segdist(P[u],P[v],P[a],P[b]))
    return best

def cubic_case(name,G,planar,seed,cap):
    G=nx.Graph(G)
    assert nx.is_connected(G) and not list(nx.bridges(G)) and all(d==3 for _,d in G.degree())
    if planar:
        ok,_=nx.check_planarity(G); assert ok
        p=nx.planar_layout(G); X=normalize([[p[n][0],p[n][1],0.] for n in G]); P={n:X[i] for i,n in enumerate(G)}
    else:
        P=None
        for trial in range(250):
            p=nx.spring_layout(G,dim=3,seed=seed+trial,iterations=700)
            X=normalize([p[n] for n in G]); Q={n:X[i] for i,n in enumerate(G)}
            if straight_clearance(G,Q)>.055: P=Q; break
        if P is None: raise RuntimeError(f'Could not find a clear 3D embedding for {name}')
    return Case(name,embedded_graph(P,[(u,v,np.linspace(P[u],P[v],100)) for u,v in G.edges()]),cap)

CASES=[
 theta_case('theta3_planar'),theta_case('theta3_bowed',True),
 cubic_case('K4',nx.complete_graph(4),True,11,.052),
 cubic_case('triangular_prism',nx.circular_ladder_graph(3),True,12,.045),
 cubic_case('cube',nx.cubical_graph(),True,13,.042),
 cubic_case('pentagonal_prism',nx.circular_ladder_graph(5),True,14,.035),
 cubic_case('dodecahedral',nx.dodecahedral_graph(),True,15,.027),
 cubic_case('K3_3',nx.complete_bipartite_graph(3,3),False,17,.032),
 cubic_case('petersen',nx.petersen_graph(),False,18,.030),
 cubic_case('heawood',nx.heawood_graph(),False,19,.024),
]
for c in CASES:
    ds=dict(c.graph.degree()); assert nx.is_connected(nx.Graph(c.graph)) and ds and set(ds.values())=={3}
    print(f'{c.name:20s} V={c.graph.number_of_nodes():2d} E={c.graph.number_of_edges():2d} degree=3')


In [ ]:
def Rxyz(a,b,c):
    a,b,c=np.deg2rad([a,b,c])
    Rx=np.array([[1,0,0],[0,np.cos(a),-np.sin(a)],[0,np.sin(a),np.cos(a)]])
    Ry=np.array([[np.cos(b),0,np.sin(b)],[0,1,0],[-np.sin(b),0,np.cos(b)]])
    Rz=np.array([[np.cos(c),-np.sin(c),0],[np.sin(c),np.cos(c),0],[0,0,1]])
    return Rz@Ry@Rx

def transform(name):
    if name=='identity': M,b=np.eye(3),np.zeros(3)
    elif name=='rotate': M,b=Rxyz(21,34,13),np.array([.04,-.03,.02])
    elif name=='affine':
        M=Rxyz(17,-23,31)@np.diag([1.08,.91,1.03])@np.array([[1,.13,0],[0,1,.09],[.05,0,1]])
        b=np.array([-.03,.04,-.02])
    else: raise ValueError(name)
    assert np.linalg.det(M)>0
    return M,b

def deform(G,name):
    M,b=transform(name); H=nx.MultiGraph()
    for n,d in G.nodes(data=True): H.add_node(n,pos=np.asarray(d['pos'])@M.T+b)
    for u,v,k,d in G.edges(keys=True,data=True): H.add_edge(u,v,pts=np.asarray(d['pts'])@M.T+b)
    return H

def trimmed(P,f=.15):
    P=np.asarray(P,float); n=max(1,int(round(f*len(P))))
    return P[n:-n] if 2*n<len(P) else P

def interior_sep(G):
    E=[(u,v,np.asarray(d['pts'],float)) for u,v,k,d in G.edges(keys=True,data=True)]
    best=np.inf
    for i,(u,v,P0) in enumerate(E):
        for a,b,Q0 in E[i+1:]:
            P,Q=(trimmed(P0),trimmed(Q0)) if {u,v}&{a,b} else (P0,Q0)
            for s in range(0,len(P),128):
                D2=np.sum((P[s:s+128,None,:]-Q[None,:,:])**2,axis=-1)
                best=min(best,float(np.sqrt(D2.min())))
    return best

def admissible(case,G,r):
    rw=r*DX; sep=interior_sep(G); limit=min(case.radius_cap,CLEARANCE_FRACTION*sep)
    return rw<=limit,rw,sep,limit

def resample(P,step):
    P=np.asarray(P,float); parts=[]
    for p,q in zip(P[:-1],P[1:]):
        n=max(2,int(np.ceil(np.linalg.norm(q-p)/step))+1); parts.append(np.linspace(p,q,n,endpoint=False))
    parts.append(P[-1:]); return np.vstack(parts)

def voxelize(G,r):
    V=np.zeros((N,N,N),bool)
    for _,_,_,d in G.edges(keys=True,data=True):
        P=resample(d['pts'],DX/3); I=np.rint((P+BOUND)/(2*BOUND)*(N-1)).astype(int); I=np.clip(I,0,N-1)
        V[I[:,0],I[:,1],I[:,2]]=1
    return dilation(V,footprint=ball(r))

def graph_stats(G):
    deg=[d for _,d in G.degree()]
    return {'V':G.number_of_nodes(),'E':G.number_of_edges(),'max_degree':max(deg,default=0),'degree_sequence':sorted(deg),'components':nx.number_connected_components(G) if G.number_of_nodes() else 0}

def recover(V):
    raw=nx.MultiGraph(skeleton_image_to_graph(skeletonize(V,method='lee')))
    H=raw.copy(); o=np.array([-BOUND]*3,float)
    for _,d in H.nodes(data=True): d['pos']=o+DX*np.asarray(d['pos'],float)
    for _,_,_,d in H.edges(keys=True,data=True): d['pts']=o+DX*np.asarray(d['pts'],float)
    raw_stats=graph_stats(H)
    H=remove_leaf_nodes(H); H=simplify_edges(H); standard_stats=graph_stats(H)
    H=contract_short_edges(H,min_length=JUNCTION_CONTRACT_VOXELS*DX,copy=False)
    H=remove_leaf_nodes(H); H=simplify_edges(H)
    H=smooth_edges(H,epsilon=SMOOTH_VOXELS*DX,copy=False)
    return H,raw_stats,standard_stats,graph_stats(H)

def abstract_ok(target,recovered):
    return nx.is_isomorphic(nx.MultiGraph(target),nx.MultiGraph(recovered))

def yamada(G):
    bad={n:d for n,d in G.degree() if d>3}
    if bad: raise ValueError(f'Yamada refused: degree > 3: {bad}')
    out=compute_yamada_polynomial(G,A,num_rotation_samples=PROJECTION_SAMPLES,crossing_warning_threshold=None,normalize=True,n_jobs=1,method='recursive',return_result=True)
    return sp.expand(out.polynomial),out.projection

def same(a,b): return sp.simplify(sp.together(sp.expand(a-b)))==0


In [ ]:
TARGETS={}
print('GROUND-TRUTH / PRE-VOXELIZATION CHECK')
for c in CASES:
    assert set(dict(c.graph.degree()).values())=={3}
    target,p=yamada(c.graph); TARGETS[c.name]=target
    print(f'TARGET {c.name:20s} V/E={c.graph.number_of_nodes()}/{c.graph.number_of_edges()} crossings={p.num_crossings:2d} Yamada={target}')
    for t in TRANSFORMS:
        H=deform(c.graph,t); assert set(dict(H.degree()).values())=={3}
        poly,_=yamada(H)
        if not same(poly,target): raise AssertionError(f'{c.name}/{t}: Yamada changed before voxelization')
print('PASS: all ground-truth graphs are exactly trivalent and all pre-voxelization deformations preserve Yamada.')


In [ ]:
payload={'schema':CACHE_SCHEMA,'N':N,'radii':RADII,'transforms':TRANSFORMS,'cases':[c.name for c in CASES],'clearance_fraction':CLEARANCE_FRACTION,'contract_voxels':JUNCTION_CONTRACT_VOXELS,'smooth_voxels':SMOOTH_VOXELS,'projection_samples':PROJECTION_SAMPLES}
SIGNATURE=hashlib.sha256(json.dumps(payload,sort_keys=True).encode()).hexdigest()[:20]
def key(c,t,r): return (SIGNATURE,c,t,int(r))

done={}
if RESUME and CHECKPOINT.exists():
    for line in CHECKPOINT.read_text().splitlines():
        if line.strip():
            x=json.loads(line)
            if x.get('signature')==SIGNATURE: done[key(x['case'],x['transform'],x['radius_vox'])]=x
records=[]; CHECKPOINT.parent.mkdir(parents=True,exist_ok=True)

for c in CASES:
  for t in TRANSFORMS:
    G=deform(c.graph,t)
    for r in RADII:
        K=key(c.name,t,r)
        if K in done:
            row=done[K]; records.append(row); print(f"CACHED     {c.name:20s} {t:8s} N={N} r={r} status={row['status']}"); continue
        ok,rw,sep,lim=admissible(c,G,r)
        row={'signature':SIGNATURE,'case':c.name,'transform':t,'resolution':N,'radius_vox':r,'radius_world':rw,'separation':sep,'limit':lim,'admissible':bool(ok),'status':None,'success':False,'raw_stats':None,'standard_stats':None,'final_stats':None,'abstract_isomorphic':None,'yamada_evaluated':False,'recovered_yamada':None,'crossings':None,'error':None}
        if not ok:
            row['status']='SKIP-GEOM'
        else:
            try:
                H,rs,ss,fs=recover(voxelize(G,r)); row['raw_stats']=rs; row['standard_stats']=ss; row['final_stats']=fs
            except Exception as exc:
                row['status']='ERROR-RECOVER'; row['error']=f'{type(exc).__name__}: {exc}'
            else:
                if fs['max_degree']>3:
                    row['status']='FAIL-DEG'
                else:
                    row['abstract_isomorphic']=bool(abstract_ok(c.graph,H))
                    if not row['abstract_isomorphic']:
                        row['status']='FAIL-GRAPH'
                    else:
                        try:
                            poly,p=yamada(H); row['yamada_evaluated']=True; row['recovered_yamada']=str(poly); row['crossings']=p.num_crossings
                        except Exception as exc:
                            row['status']='ERROR-YAMADA'; row['error']=f'{type(exc).__name__}: {exc}'
                        else:
                            row['status']='PASS' if same(poly,TARGETS[c.name]) else 'FAIL-EMBED'; row['success']=row['status']=='PASS'
        records.append(row)
        with CHECKPOINT.open('a') as f: f.write(json.dumps(row)+'\n')
        if row['status']=='SKIP-GEOM':
            print(f'SKIP-GEOM {c.name:20s} {t:8s} N={N} r={r} rw={rw:.4f} sep={sep:.4f} limit={lim:.4f}'); continue
        fs=row['final_stats']; st='V/E=?/? maxdeg=?' if fs is None else f"V/E={fs['V']}/{fs['E']} maxdeg={fs['max_degree']}"
        print(f"{row['status']:11s} {c.name:20s} {t:8s} N={N} r={r} {st} iso={row['abstract_isomorphic']} Yamada={'yes' if row['yamada_evaluated'] else 'no'}")
        if row['error']: print('    ERROR:',row['error'])
        if row['raw_stats'] and row['standard_stats']:
            print('    stages:',f"raw={row['raw_stats']['V']}/{row['raw_stats']['E']}",f"standard={row['standard_stats']['V']}/{row['standard_stats']['E']}",f"final={row['final_stats']['V']}/{row['final_stats']['E']}")

valid=[x for x in records if x['admissible']]; skipped=[x for x in records if not x['admissible']]
print('\nOVERALL'); print(f'resolution: N={N}'); print(f'candidate parameter points: {len(records)}'); print(f'geometrically inadmissible / skipped: {len(skipped)}'); print(f'admissible benchmark points: {len(valid)}')
if valid:
    p=sum(x['status']=='PASS' for x in valid); print(f'exact end-to-end recoveries: {p}/{len(valid)} = {100*p/len(valid):.2f}%')
for status in ['PASS','FAIL-DEG','FAIL-GRAPH','FAIL-EMBED','ERROR-RECOVER','ERROR-YAMADA']:
    print(f"{status:13s}: {sum(x['status']==status for x in valid)}")

for label,keyname,vals in [('PER GRAPH','case',[c.name for c in CASES]),('PER TRANSFORM','transform',TRANSFORMS),('PER RADIUS','radius_vox',RADII)]:
    print('\n'+label)
    for v in vals:
        allg=[x for x in records if x[keyname]==v]; g=[x for x in allg if x['admissible']]; skip=len(allg)-len(g)
        if not g: print(f'{str(v):20s} admissible=0 skipped={skip}'); continue
        p=sum(x['status']=='PASS' for x in g)
        C={s:sum(x['status']==s for x in g) for s in ['FAIL-DEG','FAIL-GRAPH','FAIL-EMBED','ERROR-RECOVER','ERROR-YAMADA']}
        print(f"{str(v):20s} pass={p:2d}/{len(g):2d} ({100*p/len(g):6.2f}%) skipped={skip:2d} deg={C['FAIL-DEG']:2d} graph={C['FAIL-GRAPH']:2d} embed={C['FAIL-EMBED']:2d} recover_err={C['ERROR-RECOVER']:2d} yamada_err={C['ERROR-YAMADA']:2d}")


## Interpretation

Use only **admissible** rows for the headline recovery rate.

`SKIP-GEOM` means the voxel representation is not a fair topology-preservation test because the chosen tube radius is too large relative to branch clearance.

Among admissible rows, `FAIL-DEG` diagnoses a non-trivalent recovered skeleton; `FAIL-GRAPH` diagnoses wrong abstract connectivity or edge multiplicity; and `FAIL-EMBED` is the strongest topological failure because the recovered abstract multigraph is correct but its spatial embedding has a different Yamada invariant. `ERROR-YAMADA` is reserved for genuine Yamada/projection execution errors.

The printed `raw -> standard -> final` counts show whether the cleanup removes voxel-scale artifacts or whether the reconstruction failure is already too severe to repair without changing the graph.
